# [LAB-08] 4. 다변량 분석 - 연습문제

## #01. 준비작업

### 1. 이전 분석 내용 가져오기

In [1]:
%%capture cap
%run "./[LAB-08] PBT - EDAㅣ1-EDA 시작하기(연습문제).ipynb"

### 2. 불러온 내용 확인

- 이전 분석 내역이 잘 로드 되었는지 확인한다.

In [2]:
print("종속변수 :", target)
print("종속변수 유형: " + ("연속형" if target_is_continuous else "명목형"))
print("연속형   :", continuous_cols)
print("명목형   :", nominal_cols)

display(df.head())
display(desc.head())
display(cat_desc.head())

종속변수 : charges
종속변수 유형: 연속형
연속형   : ['age', 'bmi', 'children']
명목형   : ['sex', 'smoker', 'region']


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.924
1,18,male,33.770,1,no,southeast,1725.552
2,28,male,33.000,3,no,southeast,4449.462
3,33,male,22.705,0,no,northwest,21984.471
4,32,male,28.880,0,no,northwest,3866.855


,count,mean,std,min,25%,50%,75%,max,rel_diff,rdiff_flag,iqr,upper_bound,lower_bound,upper_outliers,upper_outliers_ratio,lower_outliers,lower_outliers_ratio,outliers,outliers_ratio,skew,skew_interpret,kurt,kurt_interpret,log_need
age,1337.000,39.222,14.044,18.000,27.000,39.000,51.000,64.000,0.006,similar,24.000,87.000,-9.000,0,0.000,0,0.000,0,0.000,0.055,symmetric,-1.244,platykurtic,none
bmi,1337.000,30.663,6.100,15.960,26.290,30.400,34.700,53.130,0.009,similar,8.410,47.315,13.675,9,0.007,0,0.000,9,0.007,0.284,symmetric,-0.053,platykurtic,none
children,1337.000,1.096,1.206,0.000,0.000,1.000,2.000,5.000,0.096,similar,2.000,5.000,-3.000,0,0.000,0,0.000,0,0.000,0.937,right tail,0.201,leptokurtic,log1p
charges,1337.000,13279.121,12110.360,1121.874,4746.344,9386.161,16657.717,63770.428,0.415,diff,11911.373,34524.778,-13120.716,139,0.104,0,0.000,139,0.104,1.515,right tail,1.604,leptokurtic,log1p


,sex,smoker,region
count,1337,1337,1337
unique,2,2,4
top,male,no,southeast
freq,675,1063,364


### 3. 라이브러리 참조

In [3]:
from IPython.display import display, Markdown

## #02. 연속형 변수간의 다변량 분석

### 1. 독립변수간 상관 분석

In [5]:
# 연속형 독립변수간의 상관분석
corr = my_stats.multi_correlation(origin,columns=continuous_cols, plot=False)

# 상관분석 결과 정리
my_stats.correlation_summary(corr)

,age,bmi,children
age,1.000,0.108,0.056
bmi,0.108,1.000,0.016
children,0.056,0.016,1.000


,max-y,max-coef,count,columns
x,,,,
age,bmi,0.108,0,-
bmi,age,0.108,0,-
children,age,0.056,0,-


#### 💡 인사이트

다중공선성이 의심되는 독립변수간 관계가 발견되지 않는다.

이변량 분석에서 선정한 독립변수 후보를 최종 확정한다.

## 최종 변수 선택

| 채택여부 | 변수 | 검정방법 | 유의수준 (p) | 효과크기 | 가정점검 | 비대칭신호 | 근거 |
|---|---|---|---|---|---|---|---|
| ✅ 채택 | smoker | Mann-Whitney U test | $< 0.05$ | 미제공 | 정규성 위배 → 비모수 검정 | 해당 없음 | 집단 간 차이 유의함 (no < yes) |
| ✅ 채택 | age | Spearman | $< 0.05$ | ρ=+0.533 (중간) | 정규성 위배 / 선형성 충족 / 이상치 계수왜곡 있음 | 대칭 / 불필요 | 유의미한 상관 (\|coef\|≥0.3) |
| 🟡 후보 | children | Spearman | $< 0.05$ | ρ=+0.132 (약함) | 정규성 위배 / 선형성 위배 / 이상치 계수왜곡 없음 | 우편향 / log1p 필요 | 유의하나 관계 약함 (\|coef\|<0.3) → **모델링 투입 후 계수 유의성으로 최종 판정** |
| 🟡 후보 | bmi | Spearman | $< 0.05$ | ρ=+0.120 (약함) | 정규성 위배 / 선형성 충족 / 이상치 계수왜곡 있음 | 대칭 / 불필요 | 유의하나 관계 약함 (\|coef\|<0.3) → **모델링 투입 후 계수 유의성으로 최종 판정** |
| ❌ 제외 | region | Welch's ANOVA | $0.053$ | np2=0.007 (Negligible) | 등분산 위배 → Welch 보정 | 해당 없음 | 주효과 유의하지 않음, Games-Howell 6쌍 전부 비유의 |
| ❌ 제외 | sex | Mann-Whitney U test | $0.695$ | 미제공 | 정규성 위배 → 비모수 검정 | 해당 없음 | 집단 간 차이 유의하지 않음 (female = male) |